# 03_baselines.ipynb — Baselines (Notebook Oficial 03)

## Objetivo
Validar o pipeline ponta a ponta com baselines simples, rápidos e reproduzíveis antes do treino pesado.

## Escopo
- Ler os splits do Notebook 02
- Validar integridade do target
- Resolver carregamento de imagens sem “mágica”
- Extrair features simples de baixo custo
- Treinar baselines baratos
- Comparar com majority class
- Salvar métricas, visualizações e pacote mínimo do baseline

## Inputs
- `data/processed/train.csv`
- `data/processed/val.csv`
- `data/processed/test.csv`
- `data/processed/label_map.json`
- preferencialmente `data/processed/target_config_effective.json`
- fallback: `data/processed/target_config.json`
- `data/processed/splits_report.json`

## Outputs obrigatórios
- `data/processed/baseline_metrics.json`
- `reports/baseline_summary.md`
- `reports/baseline_confusion_matrix.png`
- `reports/baseline_examples.png`

## Outputs auxiliares
- `reports/target_contract_effective.json`
- `data/processed/baseline_package/preprocess_config.json`
- `data/processed/baseline_package/inference_config.json`
- `data/processed/baseline_package/label_map.json`

## Regras desta versão
- baseline existe para validar pipeline, não para competir com a CNN do Notebook 05
- usar o contrato effective como fonte preferencial
- executar em kernel limpo, de cima para baixo
- um único fluxo oficial de execução

In [1]:
# =========================
# 03_baselines.ipynb — Célula 01
# Setup + imports + seed + helpers básicos
# =========================

from __future__ import annotations

import os
import sys
import json
import time
import math
import random
import hashlib
import platform
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from PIL import Image, ImageOps
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

SEED = 42
IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, sort_keys=True)

def json_load(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def ensure_exists(path: Path, kind: str = "path") -> None:
    if not path.exists():
        raise FileNotFoundError(f"[ERRO] {kind} não encontrado: {path}")

def file_stat(path: Path) -> Dict[str, Any]:
    st = path.stat()
    return {
        "path": str(path),
        "size_bytes": int(st.st_size),
        "mtime_utc": datetime.fromtimestamp(st.st_mtime, tz=timezone.utc).isoformat(),
    }

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

set_global_seed(SEED)

print("OK — imports carregados.")
print("Seed fixa:", SEED)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

OK — imports carregados.
Seed fixa: 42
Python: 3.11.9
Platform: Windows-10-10.0.26200-SP0


In [2]:
# =========================
# 03_baselines.ipynb — Célula 02
# PROJECT_ROOT + paths + contratos
# =========================

def _looks_like_repo_root(p: Path) -> bool:
    return (
        (p / "data" / "raw" / "lesions" / "images").exists()
        and (p / "data" / "processed").exists()
    )

def _normalize_repo_candidate(p: Path) -> Optional[Path]:
    p = p.expanduser().resolve()
    if _looks_like_repo_root(p):
        return p
    if _looks_like_repo_root(p / "pimple"):
        return (p / "pimple").resolve()
    return None

def find_project_root_robust() -> Path:
    env_root = os.environ.get("PIMPLE_PROJECT_ROOT") or os.environ.get("PROJECT_ROOT")
    if env_root:
        normalized = _normalize_repo_candidate(Path(env_root))
        if normalized is not None:
            return normalized
        raise FileNotFoundError(
            f"[ERRO] Env PROJECT_ROOT/PIMPLE_PROJECT_ROOT aponta para {env_root}, "
            "mas não parece ser a raiz do repo pimple nem o diretório pai que contém a pasta pimple."
        )

    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        normalized = _normalize_repo_candidate(base)
        if normalized is not None:
            return normalized

    raise FileNotFoundError(
        "Não consegui localizar a raiz do repositório pimple.\n"
        "Dica: defina os.environ['PIMPLE_PROJECT_ROOT'] = r'CAMINHO_PARA_O_REPO_PIMPLE' e rode de novo."
    )

PROJECT_ROOT = find_project_root_robust()

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "lesions"
IMAGES_DIR = RAW_DIR / "images"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = PROCESSED_DIR / "train.csv"
VAL_CSV = PROCESSED_DIR / "val.csv"
TEST_CSV = PROCESSED_DIR / "test.csv"
LABEL_MAP_JSON = PROCESSED_DIR / "label_map.json"
TARGET_CONFIG_JSON = PROCESSED_DIR / "target_config.json"
TARGET_CONFIG_EFFECTIVE_JSON = PROCESSED_DIR / "target_config_effective.json"
SPLITS_REPORT_JSON = PROCESSED_DIR / "splits_report.json"

BASELINE_METRICS_JSON = PROCESSED_DIR / "baseline_metrics.json"
BASELINE_SUMMARY_MD = REPORTS_DIR / "baseline_summary.md"
BASELINE_CM_PNG = REPORTS_DIR / "baseline_confusion_matrix.png"
BASELINE_EXAMPLES_PNG = REPORTS_DIR / "baseline_examples.png"
TARGET_CONTRACT_EFFECTIVE_JSON = REPORTS_DIR / "target_contract_effective.json"

BASELINE_PACKAGE_DIR = PROCESSED_DIR / "baseline_package"
BASELINE_PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

for p, k in [
    (PROJECT_ROOT, "PROJECT_ROOT"),
    (RAW_DIR, "RAW_DIR"),
    (IMAGES_DIR, "IMAGES_DIR"),
    (PROCESSED_DIR, "PROCESSED_DIR"),
    (TRAIN_CSV, "train.csv"),
    (VAL_CSV, "val.csv"),
    (TEST_CSV, "test.csv"),
    (LABEL_MAP_JSON, "label_map.json"),
    (SPLITS_REPORT_JSON, "splits_report.json"),
]:
    ensure_exists(p, k)

TARGET_CONFIG_RESOLVED_JSON = TARGET_CONFIG_EFFECTIVE_JSON if TARGET_CONFIG_EFFECTIVE_JSON.exists() else TARGET_CONFIG_JSON
ensure_exists(TARGET_CONFIG_RESOLVED_JSON, "target_config_resolved.json")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMAGES_DIR:", IMAGES_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("REPORTS_DIR:", REPORTS_DIR)
print("TARGET_CONFIG_RESOLVED:", TARGET_CONFIG_RESOLVED_JSON)

PROJECT_ROOT: C:\Users\win\Documents\GitHub\pimple
IMAGES_DIR: C:\Users\win\Documents\GitHub\pimple\data\raw\lesions\images
PROCESSED_DIR: C:\Users\win\Documents\GitHub\pimple\data\processed
REPORTS_DIR: C:\Users\win\Documents\GitHub\pimple\reports
TARGET_CONFIG_RESOLVED: C:\Users\win\Documents\GitHub\pimple\data\processed\target_config_effective.json


In [3]:
# =========================
# 03_baselines.ipynb — Célula 03
# Load dos contratos + splits + integridade do target
# =========================

target_config = json_load(TARGET_CONFIG_RESOLVED_JSON)
label_map = json_load(LABEL_MAP_JSON)
splits_report = json_load(SPLITS_REPORT_JSON)

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

def pick(d: dict, *paths, default=None):
    for path in paths:
        cur = d
        ok = True
        for part in path.split("."):
            if isinstance(cur, dict) and part in cur:
                cur = cur[part]
            else:
                ok = False
                break
        if ok:
            if isinstance(cur, list) and len(cur) == 0:
                continue
            return cur
    return default

image_col = pick(target_config, "target_definition.image_col", "image_col", "columns.image_col")
label_cols = pick(target_config, "target_definition.label_cols", "label_cols", "columns.label_cols", default=[])
meta_cols = pick(target_config, "target_definition.meta_cols", "meta_cols", "columns.meta_cols", default=[])
mode = pick(target_config, "target_definition.mode", "mode")
target_encoding = pick(
    target_config,
    "target_definition.target_encoding",
    "target_definition.source_encoding",
    "target_encoding",
    "columns.target_encoding",
)

if not isinstance(label_cols, list) or len(label_cols) == 0:
    raise ValueError("[ERRO] target_config resolved sem label_cols válidas.")

for split_name, df_ in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing_cols = [c for c in label_cols if c not in df_.columns]
    if missing_cols:
        raise KeyError(f"[ERRO] {split_name}: colunas ausentes em label_cols: {missing_cols}")

class_names = label_map.get("index_to_label", label_cols)
n_classes = len(class_names)

ONE_HOT_ATOL = 1e-3
ONE_HOT_POLICY = "abort"

def validate_one_hot(split_name: str, df_: pd.DataFrame, label_cols: List[str], atol: float) -> Dict[str, Any]:
    y = df_[label_cols].to_numpy(dtype=np.float32)
    row_sum = y.sum(axis=1)
    ok = np.isclose(row_sum, 1.0, atol=atol)

    min_val = float(np.nanmin(y)) if y.size else float("nan")
    max_val = float(np.nanmax(y)) if y.size else float("nan")
    n_bad = int((~ok).sum())

    if min_val < -1e-3 or max_val > 1.0 + 1e-3:
        raise ValueError(f"[GATE] {split_name}: valores fora de [0,1] (min={min_val}, max={max_val}).")

    payload = {
        "split": split_name,
        "policy": ONE_HOT_POLICY,
        "atol": float(atol),
        "n_rows": int(df_.shape[0]),
        "n_rows_after_policy": int(df_.shape[0]),
        "n_bad_sum": int(n_bad),
        "bad_sum_ratio": float(n_bad / max(len(df_), 1)),
        "min_label_val": min_val,
        "max_label_val": max_val,
    }

    if n_bad > 0 and ONE_HOT_POLICY == "abort":
        raise ValueError(f"[GATE] one-hot inválido em {split_name}: n_bad_sum={n_bad}")

    return payload

integrity_train = validate_one_hot("train", train_df, label_cols, ONE_HOT_ATOL)
integrity_val = validate_one_hot("val", val_df, label_cols, ONE_HOT_ATOL)
integrity_test = validate_one_hot("test", test_df, label_cols, ONE_HOT_ATOL)

def to_y_int(df_: pd.DataFrame, label_cols: List[str]) -> np.ndarray:
    y_mat = df_[label_cols].to_numpy(dtype=np.float32)
    return np.argmax(y_mat, axis=1).astype(np.int64)

y_train = to_y_int(train_df, label_cols)
y_val = to_y_int(val_df, label_cols)
y_test = to_y_int(test_df, label_cols)

target_contract_effective = {
    "resolved_at_utc": now_utc_iso(),
    "source_target_config_json": str(TARGET_CONFIG_RESOLVED_JSON),
    "image_col": image_col,
    "label_cols": list(label_cols),
    "meta_cols": list(meta_cols) if isinstance(meta_cols, list) else [],
    "mode": mode,
    "target_encoding": target_encoding,
    "n_classes": int(n_classes),
    "class_names": list(class_names),
    "one_hot_integrity": {
        "policy": ONE_HOT_POLICY,
        "atol": ONE_HOT_ATOL,
    },
    "splits_report_meta": {
        "keys": list(splits_report.keys()) if isinstance(splits_report, dict) else str(type(splits_report)),
        "seed": splits_report.get("seed") if isinstance(splits_report, dict) else None,
    },
}
json_dump(target_contract_effective, TARGET_CONTRACT_EFFECTIVE_JSON)

print("Shapes:")
print("  train:", train_df.shape)
print("  val  :", val_df.shape)
print("  test :", test_df.shape)

print("\nTarget contract efetivo:")
print("  image_col:", image_col)
print("  n_classes:", n_classes)
print("  classes:", class_names)
print("Salvo:", TARGET_CONTRACT_EFFECTIVE_JSON)

Shapes:
  train: (7011, 12)
  val  : (1502, 12)
  test : (1502, 12)

Target contract efetivo:
  image_col: image_stem
  n_classes: 7
  classes: ['mel', 'nv', 'bcc', 'akiec', 'bkl', 'df', 'vasc']
Salvo: C:\Users\win\Documents\GitHub\pimple\reports\target_contract_effective.json


## Gate — Integridade do target

Validações mínimas:
- `label_cols` existem em todos os splits
- soma do one-hot por linha ≈ 1
- rótulos estão dentro do intervalo esperado
- baseline só continua se o contrato estiver íntegro

In [4]:
# =========================
# 03_baselines.ipynb — Célula 05
# Resolução de imagem + sanity de carregamento
# =========================

def resolve_image_path(images_dir: Path, project_root: Path, row: pd.Series, image_col: str) -> Path:
    # 1) preferir image_path, se já existir no split
    if "image_path" in row.index:
        val = row["image_path"]
        if pd.notna(val) and str(val).strip():
            cand = (project_root / str(val)).resolve()
            if cand.is_file():
                return cand

    # 2) fallback para coluna definida no contrato
    value = row[image_col] if image_col in row.index else None
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return Path("__MISSING__")

    s = str(value).strip().strip('"').strip("'")
    if not s:
        return Path("__MISSING__")

    p = Path(s)

    if p.is_absolute() and p.is_file():
        return p

    if not p.is_absolute():
        cand = (project_root / p).resolve()
        if cand.is_file():
            return cand

    cand2 = images_dir / p.name
    if cand2.is_file():
        return cand2

    stem = p.stem if p.suffix else p.name
    for ext in IMAGE_EXTS:
        cand3 = images_dir / f"{stem}{ext}"
        if cand3.is_file():
            return cand3

    return Path("__MISSING__")

def add_image_paths(df_: pd.DataFrame, split_name: str) -> pd.DataFrame:
    out = df_.copy()
    out["_image_path"] = out.apply(lambda row: resolve_image_path(IMAGES_DIR, PROJECT_ROOT, row, image_col), axis=1)

    missing_mask = out["_image_path"].astype(str) == "__MISSING__"
    missing = int(missing_mask.sum())

    if missing > 0:
        sample_cols = [c for c in ["image_path", image_col, "image_stem", "target_label"] if c in out.columns]
        sample = out.loc[missing_mask, sample_cols].head(10)
        raise FileNotFoundError(
            f"[ERRO] {split_name}: {missing} imagens não resolvidas.\n"
            f"Exemplo (até 10):\n{sample}\n\n"
            f"IMAGES_DIR={IMAGES_DIR}\nPROJECT_ROOT={PROJECT_ROOT}"
        )
    return out

train_df2 = add_image_paths(train_df, "train")
val_df2 = add_image_paths(val_df, "val")
test_df2 = add_image_paths(test_df, "test")

print("OK — resolução de imagens concluída.")
print("train:", len(train_df2), "| val:", len(val_df2), "| test:", len(test_df2))

sample_n = 5
idxs = np.random.default_rng(SEED).choice(len(train_df2), size=min(sample_n, len(train_df2)), replace=False)

print("\nSanity de carregamento (train):")
for i in idxs:
    p = Path(train_df2.iloc[int(i)]["_image_path"])
    with Image.open(p) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        print("OK load:", p.name, "| size:", im.size, "| mode:", im.mode)

OK — resolução de imagens concluída.
train: 7011 | val: 1502 | test: 1502

Sanity de carregamento (train):
OK load: ISIC_0024676.jpg | size: (600, 450) | mode: RGB
OK load: ISIC_0031660.jpg | size: (600, 450) | mode: RGB
OK load: ISIC_0027077.jpg | size: (600, 450) | mode: RGB
OK load: ISIC_0030693.jpg | size: (600, 450) | mode: RGB
OK load: ISIC_0027441.jpg | size: (600, 450) | mode: RGB


## Baseline — Features simples + Logistic Regression

Features por imagem:
- resize fixo `128x128`
- histograma RGB com `16` bins por canal
- média e desvio padrão por canal

Modelo:
- `StandardScaler`
- `LogisticRegression(solver="saga", class_weight="balanced")`

Objetivo:
- baseline barato e reproduzível
- validar o pipeline ponta a ponta antes do treino CNN

In [5]:
# =========================
# 03_baselines.ipynb — Célula 07
# Extração de features simples
# =========================

FEATURE_IMAGE_SIZE = (128, 128)
HIST_BINS = 16
FEATURE_DIM = 3 * HIST_BINS + 6

def extract_features_from_path(path: Path) -> np.ndarray:
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        im = im.resize(FEATURE_IMAGE_SIZE, resample=Image.BILINEAR)
        arr = np.asarray(im, dtype=np.uint8)

    feats = []

    # histogramas RGB em range (0, 256), incluindo 255
    for c in range(3):
        h, _ = np.histogram(arr[..., c], bins=HIST_BINS, range=(0, 256), density=True)
        feats.append(h.astype(np.float32))

    arr_f = (arr.astype(np.float32) / 255.0).reshape(-1, 3)
    mean = arr_f.mean(axis=0)
    std = arr_f.std(axis=0)

    feats.append(mean.astype(np.float32))
    feats.append(std.astype(np.float32))

    out = np.concatenate(feats, axis=0).astype(np.float32)
    if out.shape[0] != FEATURE_DIM:
        raise RuntimeError(f"FEATURE_DIM inesperado: {out.shape[0]} != {FEATURE_DIM}")
    return out

def build_feature_matrix(df_: pd.DataFrame, split_name: str) -> np.ndarray:
    paths = [Path(p) for p in df_["_image_path"].tolist()]
    iterator = paths if tqdm is None else tqdm(paths, desc=f"Features ({split_name})", total=len(paths))

    X = np.zeros((len(paths), FEATURE_DIM), dtype=np.float32)
    for i, p in enumerate(iterator):
        X[i] = extract_features_from_path(p)
    return X

X_train = build_feature_matrix(train_df2, "train")
X_val = build_feature_matrix(val_df2, "val")
X_test = build_feature_matrix(test_df2, "test")

print("X shapes:", X_train.shape, X_val.shape, X_test.shape)
print("FEATURE_DIM:", FEATURE_DIM)

Features (train):   0%|          | 0/7011 [00:00<?, ?it/s]

Features (val):   0%|          | 0/1502 [00:00<?, ?it/s]

Features (test):   0%|          | 0/1502 [00:00<?, ?it/s]

X shapes: (7011, 54) (1502, 54) (1502, 54)
FEATURE_DIM: 54


In [6]:
# =========================
# 03_baselines.ipynb — Célula 08
# Treino dos baselines + métricas principais
# =========================

def eval_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
    }

majority_class = int(pd.Series(y_train).value_counts().idxmax())
y_val_major = np.full_like(y_val, fill_value=majority_class)
y_test_major = np.full_like(y_test, fill_value=majority_class)

metrics_major_val = eval_predictions(y_val, y_val_major)
metrics_major_test = eval_predictions(y_test, y_test_major)

print("Majority baseline — val :", metrics_major_val)
print("Majority baseline — test:", metrics_major_test)

clf = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            solver="saga",
            penalty="l2",
            C=1.0,
            max_iter=3000,
            class_weight="balanced",
            n_jobs=-1,
            random_state=SEED,
        )),
    ]
)

t0 = time.time()
clf.fit(X_train, y_train)
train_time_s = time.time() - t0

y_val_pred = clf.predict(X_val)
y_test_pred = clf.predict(X_test)

metrics_lr_val = eval_predictions(y_val, y_val_pred)
metrics_lr_test = eval_predictions(y_test, y_test_pred)

print("\nLogReg baseline — train_time_s:", round(train_time_s, 2))
print("LogReg baseline — val :", metrics_lr_val)
print("LogReg baseline — test:", metrics_lr_test)

print("\nClassification report (test):")
print(classification_report(y_test, y_test_pred, target_names=class_names, digits=4))

Majority baseline — val : {'accuracy': 0.6697736351531292, 'f1_macro': 0.11460469355206197}
Majority baseline — test: {'accuracy': 0.6697736351531292, 'f1_macro': 0.11460469355206197}


c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



LogReg baseline — train_time_s: 18.83
LogReg baseline — val : {'accuracy': 0.4826897470039947, 'f1_macro': 0.31961095855672156}
LogReg baseline — test: {'accuracy': 0.48202396804260983, 'f1_macro': 0.33904869510914043}

Classification report (test):
              precision    recall  f1-score   support

         mel     0.2981    0.5569    0.3883       167
          nv     0.9492    0.4831    0.6403      1006
         bcc     0.2262    0.4935    0.3102        77
       akiec     0.1152    0.3878    0.1776        49
         bkl     0.3333    0.4000    0.3636       165
          df     0.0732    0.3529    0.1212        17
        vasc     0.2462    0.7619    0.3721        21

    accuracy                         0.4820      1502
   macro avg     0.3202    0.4909    0.3390      1502
weighted avg     0.7251    0.4820    0.5403      1502



c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [7]:
# =========================
# 03_baselines.ipynb — Célula 09
# Confusion matrix + exemplos do baseline
# =========================

cm = confusion_matrix(y_test, y_test_pred, labels=list(range(n_classes)))

fig = plt.figure(figsize=(9, 7))
ax = plt.gca()
ax.imshow(cm)

ax.set_title("Confusion Matrix — Test (LogReg baseline)")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")

ax.set_xticks(range(n_classes))
ax.set_yticks(range(n_classes))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)

for i in range(n_classes):
    for j in range(n_classes):
        ax.text(j, i, str(int(cm[i, j])), ha="center", va="center", fontsize=8)

plt.tight_layout()
fig.savefig(BASELINE_CM_PNG, dpi=160)
plt.close(fig)

print("Salvo:", BASELINE_CM_PNG)

EXAMPLES_N = 12
GRID_ROWS, GRID_COLS = 3, 4

rng = np.random.default_rng(SEED)
idxs = rng.choice(len(test_df2), size=min(EXAMPLES_N, len(test_df2)), replace=False)

proba = clf.predict_proba(X_test) if hasattr(clf, "predict_proba") else None

fig = plt.figure(figsize=(14, 10))

for k, idx in enumerate(idxs):
    row = test_df2.iloc[int(idx)]
    img_path = Path(row["_image_path"])
    true_i = int(y_test[int(idx)])
    pred_i = int(y_test_pred[int(idx)])

    with Image.open(img_path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        im_disp = im.resize((224, 224), resample=Image.BILINEAR)

    ax = plt.subplot(GRID_ROWS, GRID_COLS, k + 1)
    ax.imshow(im_disp)
    ax.axis("off")

    conf = ""
    if proba is not None:
        conf_val = float(proba[int(idx), pred_i])
        conf = f" | conf={conf_val:.2f}"

    ax.set_title(
        f"🧪 true={class_names[true_i]} | pred={class_names[pred_i]}{conf}",
        fontsize=9
    )

plt.tight_layout()
fig.savefig(BASELINE_EXAMPLES_PNG, dpi=160)
plt.close(fig)

print("Salvo:", BASELINE_EXAMPLES_PNG)

Salvo: C:\Users\win\Documents\GitHub\pimple\reports\baseline_confusion_matrix.png


C:\Users\win\AppData\Local\Temp\ipykernel_13180\52859395.py:65: UserWarning: Glyph 129514 (\N{TEST TUBE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\win\AppData\Local\Temp\ipykernel_13180\52859395.py:66: UserWarning: Glyph 129514 (\N{TEST TUBE}) missing from font(s) DejaVu Sans.
  fig.savefig(BASELINE_EXAMPLES_PNG, dpi=160)


Salvo: C:\Users\win\Documents\GitHub\pimple\reports\baseline_examples.png


In [8]:
# =========================
# 03_baselines.ipynb — Célula 10
# Salvar métricas + summary + pacote mínimo do baseline
# =========================

splits_meta = {
    "seed": splits_report.get("seed") if isinstance(splits_report, dict) else None,
    "keys": list(splits_report.keys()) if isinstance(splits_report, dict) else str(type(splits_report)),
    "splits_report_sha256": sha256_file(SPLITS_REPORT_JSON),
}

metrics_payload = {
    "created_at_utc": now_utc_iso(),
    "project_root": str(PROJECT_ROOT),
    "inputs": {
        "train_csv": file_stat(TRAIN_CSV),
        "val_csv": file_stat(VAL_CSV),
        "test_csv": file_stat(TEST_CSV),
        "label_map_json": file_stat(LABEL_MAP_JSON),
        "target_config_json": file_stat(TARGET_CONFIG_JSON) if TARGET_CONFIG_JSON.exists() else None,
        "target_config_effective_json": str(TARGET_CONFIG_EFFECTIVE_JSON) if TARGET_CONFIG_EFFECTIVE_JSON.exists() else "",
        "splits_report_json": file_stat(SPLITS_REPORT_JSON),
        "target_contract_effective_json": str(TARGET_CONTRACT_EFFECTIVE_JSON),
    },
    "splits_report_meta": splits_meta,
    "target_contract": {
        "mode": mode,
        "target_encoding": target_encoding,
        "image_col": image_col,
        "label_cols": list(label_cols),
        "meta_cols": list(meta_cols) if isinstance(meta_cols, list) else [],
        "n_classes": int(n_classes),
        "class_names": list(class_names),
    },
    "integrity": {
        "train": integrity_train,
        "val": integrity_val,
        "test": integrity_test,
    },
    "notes": {
        "target_config_effective_usage": (
            "Notebook 03 usa target_config_effective.json como contrato preferencial. "
            "A partir do Notebook 04, manter esse arquivo como fonte principal."
        )
    },
    "baselines": {
        "majority_class": {
            "val": metrics_major_val,
            "test": metrics_major_test,
        },
        "logreg_features": {
            "train_time_s": float(train_time_s),
            "val": metrics_lr_val,
            "test": metrics_lr_test,
            "confusion_matrix_test": cm.tolist(),
        },
    },
    "artifacts": {
        "baseline_metrics_json": str(BASELINE_METRICS_JSON),
        "baseline_summary_md": str(BASELINE_SUMMARY_MD),
        "baseline_examples_png": str(BASELINE_EXAMPLES_PNG),
        "baseline_confusion_matrix_png": str(BASELINE_CM_PNG),
    },
    "env": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}

json_dump(metrics_payload, BASELINE_METRICS_JSON)

preprocess_config = {
    "feature_extraction": {
        "type": "rgb_histogram_plus_stats",
        "image_size": list(FEATURE_IMAGE_SIZE),
        "hist_bins": int(HIST_BINS),
        "feature_dim": int(FEATURE_DIM),
    },
    "model": {
        "type": "sklearn_logistic_regression",
        "pipeline": [
            "StandardScaler",
            "LogisticRegression(saga, balanced)"
        ],
        "params": {
            "solver": "saga",
            "penalty": "l2",
            "C": 1.0,
            "max_iter": 3000,
            "class_weight": "balanced",
            "random_state": SEED,
        },
    },
    "seed": int(SEED),
}

inference_config = {
    "task": "classification",
    "model_version": "baseline_v1",
    "n_classes": int(n_classes),
    "class_names": list(class_names),
}

json_dump(preprocess_config, BASELINE_PACKAGE_DIR / "preprocess_config.json")
json_dump(inference_config, BASELINE_PACKAGE_DIR / "inference_config.json")
json_dump(label_map, BASELINE_PACKAGE_DIR / "label_map.json")

policy = str(integrity_train.get("policy", "unknown"))
atol = integrity_train.get("atol", "n/a")

def _ratio(payload: Dict[str, Any]) -> float:
    n_bad = float(payload.get("n_bad_sum", 0))
    n_rows = float(payload.get("n_rows_after_policy", payload.get("n_rows", 1)))
    return float(payload.get("bad_sum_ratio", n_bad / max(n_rows, 1.0)))

tr_bad = int(integrity_train.get("n_bad_sum", 0))
va_bad = int(integrity_val.get("n_bad_sum", 0))
te_bad = int(integrity_test.get("n_bad_sum", 0))

tr_ratio = _ratio(integrity_train)
va_ratio = _ratio(integrity_val)
te_ratio = _ratio(integrity_test)

tr_rows = int(integrity_train.get("n_rows_after_policy", integrity_train.get("n_rows", 0)))
va_rows = int(integrity_val.get("n_rows_after_policy", integrity_val.get("n_rows", 0)))
te_rows = int(integrity_test.get("n_rows_after_policy", integrity_test.get("n_rows", 0)))

seed_in_splits = splits_meta.get("seed", "n/a")
hash_splits = splits_meta["splits_report_sha256"]

summary_md = f"""# Baseline Summary - pimple (Notebook 03)

Created (UTC): {metrics_payload["created_at_utc"]}
Seed (notebook): {SEED}
Seed (splits_report): {seed_in_splits}
splits_report_sha256: {hash_splits}

## Target contract (effective)
- mode: {mode}
- target_encoding: {target_encoding}
- image_col: {image_col}
- n_classes: {n_classes}
- classes: {", ".join(class_names)}

## Effective config
- target_config_resolved.json: {TARGET_CONFIG_RESOLVED_JSON}
  - Recommendation: use this file from Notebook 04 onward.

## Target integrity (one-hot)
- policy: {policy} (atol={atol})
- train: rows={tr_rows} | n_bad_sum={tr_bad} | ratio={tr_ratio:.6f}
- val: rows={va_rows} | n_bad_sum={va_bad} | ratio={va_ratio:.6f}
- test: rows={te_rows} | n_bad_sum={te_bad} | ratio={te_ratio:.6f}

## Metrics (val / test)

| Baseline | Accuracy (val) | F1 macro (val) | Accuracy (test) | F1 macro (test) |
|---|---:|---:|---:|---:|
| Majority | {metrics_major_val["accuracy"]:.4f} | {metrics_major_val["f1_macro"]:.4f} | {metrics_major_test["accuracy"]:.4f} | {metrics_major_test["f1_macro"]:.4f} |
| LogReg (features) | {metrics_lr_val["accuracy"]:.4f} | {metrics_lr_val["f1_macro"]:.4f} | {metrics_lr_test["accuracy"]:.4f} | {metrics_lr_test["f1_macro"]:.4f} |

## Artifacts
- data/processed/baseline_metrics.json
- reports/baseline_summary.md
- reports/baseline_examples.png
- reports/baseline_confusion_matrix.png
- data/processed/baseline_package/preprocess_config.json
- data/processed/baseline_package/inference_config.json
- data/processed/baseline_package/label_map.json
"""

BASELINE_SUMMARY_MD.write_text(summary_md, encoding="utf-8")

print("Salvo:", BASELINE_METRICS_JSON)
print("Salvo:", BASELINE_SUMMARY_MD)
print("Salvo:", BASELINE_PACKAGE_DIR / "preprocess_config.json")
print("Salvo:", BASELINE_PACKAGE_DIR / "inference_config.json")
print("Salvo:", BASELINE_PACKAGE_DIR / "label_map.json")

Salvo: C:\Users\win\Documents\GitHub\pimple\data\processed\baseline_metrics.json
Salvo: C:\Users\win\Documents\GitHub\pimple\reports\baseline_summary.md
Salvo: C:\Users\win\Documents\GitHub\pimple\data\processed\baseline_package\preprocess_config.json
Salvo: C:\Users\win\Documents\GitHub\pimple\data\processed\baseline_package\inference_config.json
Salvo: C:\Users\win\Documents\GitHub\pimple\data\processed\baseline_package\label_map.json


In [9]:
# =========================
# 03_baselines.ipynb — Célula 11
# Gate final
# =========================

gate_checks = {
    "baseline_metrics_exists": BASELINE_METRICS_JSON.exists(),
    "baseline_summary_exists": BASELINE_SUMMARY_MD.exists(),
    "baseline_confusion_matrix_exists": BASELINE_CM_PNG.exists(),
    "baseline_examples_exists": BASELINE_EXAMPLES_PNG.exists(),
    "target_contract_effective_exists": TARGET_CONTRACT_EFFECTIVE_JSON.exists(),
    "baseline_preprocess_config_exists": (BASELINE_PACKAGE_DIR / "preprocess_config.json").exists(),
    "baseline_inference_config_exists": (BASELINE_PACKAGE_DIR / "inference_config.json").exists(),
    "baseline_label_map_exists": (BASELINE_PACKAGE_DIR / "label_map.json").exists(),
    "train_rows_positive": len(train_df) > 0,
    "val_rows_positive": len(val_df) > 0,
    "test_rows_positive": len(test_df) > 0,
    "feature_dim_ok": X_train.shape[1] == FEATURE_DIM,
}

gate_pass = all(gate_checks.values())

print("=== OK — Artefatos salvos ===")
print("baseline_metrics_json      :", BASELINE_METRICS_JSON)
print("baseline_summary_md        :", BASELINE_SUMMARY_MD)
print("baseline_confusion_matrix  :", BASELINE_CM_PNG)
print("baseline_examples_png      :", BASELINE_EXAMPLES_PNG)
print("target_contract_effective  :", TARGET_CONTRACT_EFFECTIVE_JSON)
print("baseline_package_dir       :", BASELINE_PACKAGE_DIR)

print("\n=== Gate final ===")
for k, v in gate_checks.items():
    print(f"- {k}: {v}")
print("STATUS:", "PASS" if gate_pass else "FAIL")

=== OK — Artefatos salvos ===
baseline_metrics_json      : C:\Users\win\Documents\GitHub\pimple\data\processed\baseline_metrics.json
baseline_summary_md        : C:\Users\win\Documents\GitHub\pimple\reports\baseline_summary.md
baseline_confusion_matrix  : C:\Users\win\Documents\GitHub\pimple\reports\baseline_confusion_matrix.png
baseline_examples_png      : C:\Users\win\Documents\GitHub\pimple\reports\baseline_examples.png
target_contract_effective  : C:\Users\win\Documents\GitHub\pimple\reports\target_contract_effective.json
baseline_package_dir       : C:\Users\win\Documents\GitHub\pimple\data\processed\baseline_package

=== Gate final ===
- baseline_metrics_exists: True
- baseline_summary_exists: True
- baseline_confusion_matrix_exists: True
- baseline_examples_exists: True
- target_contract_effective_exists: True
- baseline_preprocess_config_exists: True
- baseline_inference_config_exists: True
- baseline_label_map_exists: True
- train_rows_positive: True
- val_rows_positive: True
